#### Build Machine Learning Model

In [ ]:
import ast
import os

import neurokit2 as nk
import numpy as np
import pandas as pd
import wfdb

In [ ]:
# create mock dataset

# connect dataset
# PTB-XL folder in the project root (git-ignored; a symlink to the dataset works)
ecg_db_path = os.path.join("..", "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
sampling_rate = 100  # 100Hz -> filename_lr (records100/), 500Hz -> filename_hr (records500/)

# load metadata; scp_codes is stored as a dict string
Y = pd.read_csv(os.path.join(ecg_db_path, "ptbxl_database.csv"), index_col="ecg_id")
Y.scp_codes = Y.scp_codes.apply(ast.literal_eval)

# recording_date is stored as text -> parse to datetime (the only timestamp; .hea headers have none)
Y.recording_date = pd.to_datetime(Y.recording_date, format="%Y-%m-%d %H:%M:%S")

# ages over 89 are stored as 300 for privacy -> treat as missing
Y.loc[Y.age == 300, "age"] = np.nan

# build the full path to each 100Hz record (wfdb expects the path without the .dat/.hea extension)
Y["path_100hz"] = Y.filename_lr.apply(lambda f: os.path.join(ecg_db_path, f))
patient_cols = ["patient_id", "recording_date", "age", "sex", "height", "weight", "pacemaker",
                "scp_codes", "heart_axis", "infarction_stadium1", "strat_fold", "filename_lr"]
Y[patient_cols].head()

In [ ]:
# map scp_codes to diagnostic superclasses (NORM, MI, STTC, CD, HYP)
agg_df = pd.read_csv(os.path.join(ecg_db_path, "scp_statements.csv"), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]  # keep only diagnostic statements (drops form/rhythm codes)


def aggregate_diagnostic(scp_codes):
    return sorted({agg_df.loc[code, "diagnostic_class"] for code in scp_codes if code in agg_df.index})


Y["diagnostic_superclass"] = Y.scp_codes.apply(aggregate_diagnostic)
Y[["scp_codes", "diagnostic_superclass"]].head()

In [ ]:
# check if the same patient has multiple ECGs
ecgs_per_patient = Y.groupby("patient_id").size()
print(f"{len(Y)} ECGs from {ecgs_per_patient.size} patients; "
      f"{(ecgs_per_patient > 1).sum()} patients have more than one ECG")

# strat_fold keeps all ECGs of a patient in one fold -> no patient leakage between train/val/test
assert Y.groupby("patient_id").strat_fold.nunique().max() == 1
ecgs_per_patient.value_counts().sort_index().rename_axis("ECGs per patient").to_frame("patients")

In [ ]:
# order each patient's recordings by date/time
Y["recording_day"] = Y.recording_date.dt.date
Y["recording_time"] = Y.recording_date.dt.time
Y = Y.sort_values(["patient_id", "recording_date"])
Y["visit_number"] = Y.groupby("patient_id").cumcount() + 1                      # 1 = first ECG of that patient
Y["time_since_prev"] = Y.groupby("patient_id").recording_date.diff()           # NaT for the first ECG
Y["days_since_first"] = (Y.recording_date - Y.groupby("patient_id").recording_date.transform("min")).dt.days
Y = Y.sort_index()  # back to ecg_id order so rows line up with X

# example: all recordings of the patient with the most ECGs, in time order
pid = ecgs_per_patient.idxmax()
Y[Y.patient_id == pid].sort_values("recording_date")[
    ["patient_id", "recording_date", "visit_number", "time_since_prev", "days_since_first", "diagnostic_superclass"]]

In [ ]:
# for patients with exactly two ECGs: time difference between the two recordings
two_ecg = Y[Y.patient_id.isin(ecgs_per_patient[ecgs_per_patient == 2].index)]
gap = two_ecg[two_ecg.visit_number == 2].set_index("patient_id").time_since_prev
print(f"{len(gap)} patients with two ECGs")
print(gap.describe())

gap_days = gap.dt.total_seconds() / 86400
bins = [-1e-9, 1 / 24, 1, 7, 30, 90, 365, np.inf]
labels = ["< 1 hour", "1 hour - 1 day", "1 - 7 days", "1 - 4 weeks", "1 - 3 months", "3 - 12 months", "> 1 year"]
pd.cut(gap_days, bins, labels=labels).value_counts().sort_index().rename_axis("time between ECGs").to_frame("patients")

In [ ]:
def load_raw_data(paths):
    """Read each record's signal -> array of shape (n_records, 1000, 12)."""
    return np.array([wfdb.rdsamp(p)[0] for p in paths])


X = load_raw_data(Y.path_100hz)
X.shape

In [ ]:
# check signal length: every record should be 10 s at 100Hz -> 1000 samples x 12 leads
n_records, sig_len, n_leads = X.shape
print(f"{n_records} records, {sig_len} samples ({sig_len / sampling_rate:.0f} s at {sampling_rate}Hz), {n_leads} leads")
assert (sig_len, n_leads) == (10 * sampling_rate, 12)

In [ ]:
# HRV for patients with two ECGs (10 s recordings -> ultra-short-term time-domain HRV only)
LEAD_II = 1  # lead order: I, II, III, AVR, AVL, AVF, V1-V6


def compute_hrv(ecg, fs=sampling_rate):
    """R-peaks from NeuroKit2 -> RR intervals (ms) -> time-domain HRV."""
    try:
        clean = nk.ecg_clean(ecg, sampling_rate=fs)
        _, info = nk.ecg_peaks(clean, sampling_rate=fs, correct_artifacts=True)
        rr = np.diff(info["ECG_R_Peaks"]) / fs * 1000
    except Exception:
        rr = np.array([])
    if len(rr) < 5:  # too few beats for a meaningful HRV value
        return {"n_beats": len(rr) + 1 if len(rr) else 0}
    return {
        "n_beats": len(rr) + 1,
        "mean_rr_ms": rr.mean(),
        "mean_hr_bpm": 60000 / rr.mean(),
        "sdnn_ms": rr.std(ddof=1),
        "rmssd_ms": np.sqrt(np.mean(np.diff(rr) ** 2)),
        "pnn50_pct": np.mean(np.abs(np.diff(rr)) > 50) * 100,
    }


hrv_ids = two_ecg.sort_values(["patient_id", "recording_date"]).index
hrv_rows = X[Y.index.get_indexer(hrv_ids), :, LEAD_II]  # X rows are in Y (ecg_id) order
hrv = pd.DataFrame([compute_hrv(ecg) for ecg in hrv_rows], index=hrv_ids)

# HRV is only meaningful in sinus rhythm; AFIB / extra beats inflate SDNN and RMSSD
Y["sinus_rhythm"] = Y.scp_codes.apply(lambda codes: "SR" in codes)
hrv_df = Y.loc[hrv_ids, ["patient_id", "visit_number", "recording_date", "time_since_prev",
                         "age", "sex", "diagnostic_superclass", "sinus_rhythm"]].join(hrv)
hrv_df.to_csv("hrv_two_ecg_patients.csv")
print(f"{len(hrv_df)} ECGs, {hrv_df.sdnn_ms.isna().sum()} without enough beats for HRV, "
      f"{hrv_df.sinus_rhythm.sum()} in sinus rhythm")
hrv_df.head(10)

In [ ]:
# compare HRV between the 1st and 2nd ECG of each patient (pNN50 left out)
from scipy.stats import wilcoxon

hrv_metrics = ["mean_rr_ms", "mean_hr_bpm", "sdnn_ms", "rmssd_ms"]
wide = hrv_df.pivot(index="patient_id", columns="visit_number", values=hrv_metrics + ["sinus_rhythm"])
wide.columns = [f"{m}_{v}" for m, v in wide.columns]

hrv_compare = pd.DataFrame({
    "days_between": hrv_df[hrv_df.visit_number == 2].set_index("patient_id").time_since_prev.dt.days,
    "both_sinus": wide.sinus_rhythm_1.astype(bool) & wide.sinus_rhythm_2.astype(bool),
})
for m in hrv_metrics:
    first, second = wide[f"{m}_1"].astype(float), wide[f"{m}_2"].astype(float)
    hrv_compare[f"{m}_1"] = first
    hrv_compare[f"{m}_2"] = second
    hrv_compare[f"{m}_change"] = second - first  # 2nd ECG minus 1st
hrv_compare.to_csv("hrv_compare_two_ecg.csv")

# paired summary, only patients in sinus rhythm on both ECGs with HRV on both
pairs = hrv_compare[hrv_compare.both_sinus].dropna(subset=[f"{m}_change" for m in hrv_metrics])
summary = pd.DataFrame({
    m: {
        "median_1st": pairs[f"{m}_1"].median(),
        "median_2nd": pairs[f"{m}_2"].median(),
        "median_change": pairs[f"{m}_change"].median(),
        "median_abs_change": pairs[f"{m}_change"].abs().median(),
        "wilcoxon_p": wilcoxon(pairs[f"{m}_1"], pairs[f"{m}_2"]).pvalue,
    }
    for m in hrv_metrics
}).T.round(3)
print(f"{len(hrv_compare)} patients, {len(pairs)} in sinus rhythm on both ECGs")
summary